In [ ]:
import ee
import geemap
from pathlib import Path
from datetime import datetime

# ---- Config -------------------------------------------------
MAX_CLOUD   = 5
START_YEAR  = 2020
END_YEAR    = 2025
START_MONTH = 6
END_MONTH   = 8
OUT_DIR     = Path("Heatwave_risk_KZ/data/LST")

# ---- Initialise GEE -----------------------------------------
ee.Initialize(project='your-gee-project-id')

# ---- Astana bounding box ------------------------------------
astana = ee.Geometry.Rectangle([71.00, 50.85, 71.90, 51.45])

# ---- Cloud mask + LST (°C) ----------------------------------
def process_l8l9(image):
    qa   = image.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 3).eq(0)
              .And(qa.bitwiseAnd(1 << 4).eq(0)))
    lst  = (image.select('ST_B10')
                 .multiply(0.00341802).add(149.0).subtract(273.15)
                 .rename('LST'))
    return (lst.updateMask(mask)
               .copyProperties(image, ['system:time_start',
                                       'LANDSAT_PRODUCT_ID',
                                       'SPACECRAFT_ID',
                                       'CLOUD_COVER']))

# ---- Build collection ---------------------------------------
filtered = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
      .filterBounds(astana)
      .filter(ee.Filter.calendarRange(START_YEAR, END_YEAR,   'year'))
      .filter(ee.Filter.calendarRange(START_MONTH, END_MONTH, 'month'))
      .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
      .sort('system:time_start')
      .map(process_l8l9)
)

n = filtered.size().getInfo()
print(f"Images found: {n}  (cloud < {MAX_CLOUD}%, June–Aug {START_YEAR}–{END_YEAR})")

# ---- Download — one file per image with explicit filename ---
OUT_DIR.mkdir(parents=True, exist_ok=True)

image_list = filtered.toList(n)
print(f"Downloading {n} images to {OUT_DIR.resolve()} ...")

for i in range(n):
    img   = ee.Image(image_list.get(i))
    t_ms  = img.get('system:time_start').getInfo()
    t     = datetime.utcfromtimestamp(t_ms / 1000)
    fname = t.strftime('LST.%Y%m%d_%H%M%S')
    out_path = OUT_DIR / f"{fname}.tif"

    if out_path.exists():
        print(f"  [{i+1}/{n}] Skipping (already exists): {fname}.tif")
        continue

    print(f"  [{i+1}/{n}] Downloading: {fname}.tif")
    geemap.ee_export_image(
        img.select('LST'),
        filename=str(out_path),
        scale=30,
        region=astana,
        file_per_band=False,
    )

print("Done. Files are ready in LST/ — run the heatwave notebook next.")